# Assignment 2 — Image dehazing: 4 recent models (2024–2026) × 3 datasets

**Models:** DEA-Net (TIP 2024) · ConvIR (TPAMI 2024) · CoA (CVPR 2025) · BiLaLoRA (CVPR 2026) (official code + official pretrained weights)  
**Datasets:** RESIDE-SOTS indoor (500 pairs) · O-HAZE (45 pairs, test = 41–45) · NH-HAZE (55 pairs, test = 51–55)  
**Metrics:** PSNR, SSIM, MAE, LPIPS (full-reference), MUSIQ (no-reference), inference time and parameters, all computed with the same code for every model.

Run the cells from top to bottom. Every step can be re-run safely; finished downloads and predictions are skipped.

## 0. Install packages (first time only)

In [ ]:
import importlib.util, subprocess, sys, shutil
if importlib.util.find_spec("torch") is None:
    if shutil.which("nvidia-smi"):   # NVIDIA GPU -> CUDA build
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision",
                               "--index-url", "https://download.pytorch.org/whl/cu126"])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("packages ready")

## 1. Setup

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path(r"C:\Users\lab\Desktop\dehaze")
if not (ROOT / "dehaze_lib").exists():
    ROOT = Path.cwd()          # notebook opened from the project folder elsewhere
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import torch, pandas as pd
from IPython.display import display, Markdown
from dehaze_lib import config as C
from dehaze_lib.download import download_datasets, download_weights, weights_status
from dehaze_lib.data import organize_all, get_pairs
from dehaze_lib.models import run_inference
from dehaze_lib.evaluate import evaluate_all, plot_results, plot_gain, plot_perceptual, show_examples
from dehaze_lib.perceptual import evaluate_perceptual

print("Project:", C.ROOT)
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, slow)")
print("Models  :", C.MODELS)
print("Datasets:", C.DATASETS)

## 2. Download the datasets (≈1.3 GB) and the official weights

In [ ]:
download_datasets()

In [ ]:
download_weights()
pd.Series(weights_status(), name="downloaded")

## 3. Organize into `datasets/<name>/{hazy,gt}`
RESIDE-SOTS indoor has 50 clear scenes × 10 haze levels = 500 hazy images, so each hazy image is matched to its scene's GT.
O-HAZE is downscaled to a maximum side of 1600 px (hazy and GT by the same factor; see `C.MAX_SIDE`).

In [ ]:
organize_all();

## 4. Quick test (3 images per dataset)
Checks that every model loads and runs before starting the long run.

In [ ]:
run_inference(limit=3);

## 5. Full inference
Writes `predictions/<dataset>/<model>/<image>.png`. Takes roughly 15–40 min on a GPU and several hours on CPU.
If you get out-of-memory errors, add `tile=512`.

In [ ]:
log = run_inference()          # e.g. run_inference(models=["DEA-Net"], datasets=["O_HAZE"], tile=512)
pd.read_csv(C.RESULTS / "inference_log.csv")

## 6. Evaluation: PSNR / SSIM / MAE

In [ ]:
summary = evaluate_all()
cols = ["Dataset", "Model", "Checkpoint", "N", "PSNR", "SSIM", "MAE", "Sec_per_image", "Params_M"]  # *_test = official test splits
display(summary[[c for c in cols if c in summary]].round(4))

In [ ]:
for metric in ["PSNR", "SSIM"]:
    display(Markdown(f"**{metric}** (rows = model, columns = dataset)"))
    display(pd.read_csv(C.RESULTS / f"table_{metric}.csv", index_col=0).round(4))

## 6b. Perceptual metrics: MUSIQ (no-reference) and LPIPS
Downloads the MUSIQ (KonIQ) and LPIPS-Alex models once. The ground-truth images are also scored with MUSIQ as a reference.

In [ ]:
summary = evaluate_perceptual()
cols = ["Dataset", "Model", "PSNR", "SSIM", "LPIPS", "MUSIQ"]
display(summary[[c for c in cols if c in summary]].round(4))

In [ ]:
for metric in ["LPIPS", "MUSIQ"]:
    display(Markdown(f"**{metric}** (rows = model, columns = dataset)"))
    display(pd.read_csv(C.RESULTS / f"table_{metric}.csv", index_col=0).round(4))

## 7. Figures

In [ ]:
plot_results(summary)
plot_gain(summary)
plot_perceptual(summary);

In [ ]:
# first official test image for O-HAZE / NH-HAZE (ConvIR's checkpoints were trained on the other images)
for ds in C.DATASETS:
    ids = next((ids for b, ids in C.TEST_SUBSETS.values() if b == ds), None)
    pairs, _ = get_pairs(ds)
    idx = next((i for i, (sid, _, _) in enumerate(pairs) if ids and sid.split("_")[0] in ids), 0)
    show_examples(ds, idx)      # change idx to see other images

## 8. Notes for the report
* **Checkpoints:** DEA-Net uses its RESIDE-ITS weights on SOTS-indoor and its OTS (outdoor) weights on O-HAZE / NH-HAZE (no real-haze weights released, so this is cross-dataset).
  ConvIR-B uses its official per-benchmark checkpoints (ITS, O-HAZE, NH-HAZE); the O/NH ones were trained on those datasets' training images, so only the official **test splits** (O-HAZE 41–45, NH-HAZE 51–55) are a fair comparison.
  CoA and BiLaLoRA each release one real-world model, used on every set.
* **Fair comparison:** same test images, same pre-processing, same metric code (scikit-image PSNR/SSIM, `lpips` LPIPS-Alex, `pyiqa` MUSIQ).
* **Differences from the papers** are expected: SSIM implementations (MATLAB vs Python), O-HAZE resizing to 1600 px.
* CoA and BiLaLoRA report only no-reference scores (MUSIQ, FADE, BIQME), which is why MUSIQ and LPIPS are included.
* All files: `results/summary_all.csv`, `results/summary.md`, `results/table_*.csv`, `results/per_image*/`, `results/figures/`.